### What we are building



A state machine based secure authentication pipeline

Flow:

<pre>
Detect Person
    ↓
Recognize Identity
    ↓
Check Liveness
    ↓
Ask: Blink
    ↓
Ask: Turn Left
    ↓
Ask: Turn Right
    ↓
Authorize
</pre>

In [40]:
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
import torch

In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device, keep_all=True)

resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

In [42]:
database = {}

for person in os.listdir('faces/single'):
    person_embeddings = []
    
    for file in os.listdir(f'faces/single/{person}'):
        img = Image.open(os.path.join(f'faces/single/{person}', file))
        
        face = mtcnn(img)
        
        if face is None:
            continue
        
        # If multiple faces are detected take first one:
        if face.ndim == 4:
            face = face[0].unsqueeze(0)
        
        face = face.to(device)
        
        with torch.no_grad():
            embedding = resnet(face)
            
        # Normalize embedding
        embedding /= embedding.norm(dim=1, keepdim=True)
        
        person_embeddings.append(embedding)
        
    if len(person_embeddings) > 0:
        # Average embeddings
        database[person] = torch.mean(torch.cat(person_embeddings), dim=0, keepdim=True)
            
print("Database Loaded: ", list(database.keys()))

Database Loaded:  ['abhishek_bachchan', 'amitabh_bachchan', 'chirag', 'jaya_bachchan', 'shail']


In [43]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

In [44]:
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(eye_points):
    A = euclidean(eye_points[1], eye_points[5])
    B = euclidean(eye_points[2], eye_points[4])
    C = euclidean(eye_points[0], eye_points[3])

    ear = (A + B) / (2.0 * C)
    return ear

In [45]:
AUTH_DURATION = 15
EAR_THRESHOLD = 0.2
BLINK_FRAMES = 1
HEAD_THRESHOLD = 40

DETECT_EVERY = 5   # run detection every N frames
DOWNSCALE_W = 320
DOWNSCALE_H = 240

active_sessions = {}          # { person: expiry_timestamp }
current_auth_target = None    # Only one person at a time

blink_counter = 0
auth_state = "IDLE"

stored_detections = []
prev_time = time.time()
fps = 0
frame_count = 0

In [46]:
def recognize_faces(frame, database):

    global stored_detections

    small = cv2.resize(frame, (DOWNSCALE_W, DOWNSCALE_H))
    small_rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(small_rgb)

    faces = mtcnn(img)

    if faces is None:
        stored_detections = []
        time.sleep(0.25)
        return []

    if faces.ndim == 3:
        faces = faces.unsqueeze(0)

    faces = faces.to(device)

    with torch.no_grad():
        embeddings = resnet(faces)

    embeddings /= embeddings.norm(dim=1, keepdim=True)

    boxes, _ = mtcnn.detect(img)

    if boxes is None:
        stored_detections = []
        time.sleep(0.25)
        return []

    scale_x = frame.shape[1] / DOWNSCALE_W
    scale_y = frame.shape[0] / DOWNSCALE_H

    detections = []

    for i, emb in enumerate(embeddings):

        emb = emb.unsqueeze(0)

        best_match = None
        max_sim = -1

        for name, db_emb in database.items():
            sim = F.cosine_similarity(emb, db_emb).item()
            if sim > max_sim:
                max_sim = sim
                best_match = name

        person = best_match if max_sim > 0.75 else None

        x1, y1, x2, y2 = boxes[i]
        x1 = int(x1 * scale_x)
        y1 = int(y1 * scale_y)
        x2 = int(x2 * scale_x)
        y2 = int(y2 * scale_y)

        detections.append((person, (x1,y1,x2,y2)))

    stored_detections = detections
    return detections

In [57]:
def run_liveness(frame, box, person_name):

    global blink_counter, auth_state, current_auth_target

    x1,y1,x2,y2 = box
    face_crop = frame[y1:y2, x1:x2]

    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=face_rgb
    )

    timestamp = int(time.time()*1000)
    result = landmarker.detect_for_video(mp_image, timestamp)

    if not result.face_landmarks:
        return frame

    h, w, _ = face_crop.shape
    landmarks = result.face_landmarks[0]

    # ---------- HEAD ----------
    nose = landmarks[1]
    nose_x = int(nose.x * w)
    center_x = w // 2
    offset = nose_x - center_x

    if offset > HEAD_THRESHOLD:
        head_dir = "LEFT"
    elif offset < -HEAD_THRESHOLD:
        head_dir = "RIGHT"
    else:
        head_dir = "CENTER"

    # ---------- EYES ----------
    left_idx = [33,160,158,133,153,144]
    right_idx = [362,385,387,263,373,380]

    left_pts = []
    right_pts = []

    for idx in left_idx:
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        left_pts.append((x,y))
        cv2.circle(face_crop, (x,y), 2, (0,255,0), -1)

    for idx in right_idx:
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        right_pts.append((x,y))
        cv2.circle(face_crop, (x,y), 2, (0,255,0), -1)

    ear = (calculate_ear(left_pts) + calculate_ear(right_pts)) / 2
    
    cv2.putText(frame, f"EAR: {ear:.3f}", (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (255, 0, 0), 2)

    blink_detected = False

    if ear < EAR_THRESHOLD:
        blink_counter += 1
    else:
        if blink_counter >= BLINK_FRAMES:
            blink_detected = True
        blink_counter = 0

    # ---------- STATE MACHINE ----------
    command_text = ""

    if auth_state == "IDLE":
        auth_state = "BLINK"

    if auth_state == "BLINK":
        command_text = "Please Blink"
        if blink_detected:
            auth_state = "LEFT"

    elif auth_state == "LEFT":
        command_text = "Turn Head Left"
        if head_dir == "LEFT":
            auth_state = "RIGHT"

    elif auth_state == "RIGHT":
        command_text = "Turn Head Right"
        if head_dir == "RIGHT":
            active_sessions[person_name] = time.time() + AUTH_DURATION
            current_auth_target = None
            auth_state = "IDLE"

    # ---------- DRAW COMMAND TEXT ----------
    cv2.putText(frame,
                command_text,
                (x1, y1 - 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2)

    return frame

In [58]:
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()
    frame = cv2.resize(frame, (640,480))

    if not ret:
        break

    frame_count += 1

    # Detect every N frames
    if frame_count % DETECT_EVERY == 0:
        detections = recognize_faces(frame, database)
    else:
        detections = stored_detections

    current_time = time.time()

    for person, box in detections:

        x1,y1,x2,y2 = box

        if person:

            # Session active?
            if person in active_sessions:
                expiry = active_sessions[person]

                if current_time < expiry:
                    remaining = int(expiry - current_time)
                    label = f"{person} ({remaining}s)"
                    color = (0,255,0)
                else:
                    del active_sessions[person]
                    label = f"{person} - Reauth"
                    color = (0,0,255)

            else:
                label = f"{person} - Verify"
                color = (0,255,255)

                if current_auth_target is None:
                    current_auth_target = person

        else:
            label = "Unknown"
            color = (0,0,255)

        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cv2.putText(frame, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, color, 2)

        # Run liveness only for active target
        if person == current_auth_target:
            frame = run_liveness(frame, box, person)

    # FPS
    new_time = time.time()
    fps = 0.9 * fps + 0.1 * (1 / (new_time - prev_time))
    prev_time = new_time

    cv2.putText(frame, f"FPS: {int(fps)}",
                (20,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,255,0), 2)

    cv2.imshow("Optimized Secure System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()